# 从多层感知机到卷积神经网络

MLP 对于表格类数据处理很好，但是对于图像处理可以算得上是一种灾难。
对于一个大小为 32MB 的图像，它大概有 1118 万个像素点。
即使我们使用只有一层隐藏层，维度为 100 的 MLP，也需要 100 * 1118 万 = 11.18 亿个参数。
想训练这个模型的开销和时间是不现实的。
即使我们将分辨率缩小到 10 万个像素点，使用 100 个隐藏单元的隐藏层也不足以学习到良好的图像特征。
此外，学习如此多的参数还需要换手机大量的数据.

目前，人类和机器都能很好区分猫和狗：这是因为图像中本就拥有丰富的结构，而这些结构可以被人类和机器学习模型使用。
*卷积神经网络*(Convolutional Neural Networks, CNN) 是机器学习利用自然图像中一些已知结构的创造性方法。


## 不变性
想象一下，假设我们想从一张图片中找到某个物体。
合理的假设是：无论哪种方法找到这个物体，都应该和物体的位置无关。
理想情况下，我们的系统应该能够利用常识：猪通常不在天上飞，飞机通常不在水里游泳。
但是，如果一只猪出现在图片顶部，我们还是应该认出它。
我们可以从儿童游戏”沃尔多在哪里”（ :numref:`img_waldo`）中得到灵感：
在这个游戏中包含了许多充斥着活动的混乱场景，而沃尔多通常潜伏在一些不太可能的位置，读者的目标就是找出他。
尽管沃尔多的装扮很有特点，但是在眼花缭乱的场景中找到他也如大海捞针。
然而沃尔多的样子并不取决于他潜藏的地方，因此我们可以使用一个“沃尔多检测器”扫描图像。
该检测器将图像分割成多个区域，并为每个区域包含沃尔多的可能性打分。
卷积神经网络正是将*空间不变性*（spatial invariance）的这一概念系统化，从而基于这个模型使用较少的参数来学习有用的表示。

![沃尔多游戏示例图。](../img/where-wally-walker-books.jpg)
:width:`400px`
:label:`img_waldo`

现在，我们将上述想法总结一下，从而帮助我们设计适合于计算机视觉的神经网络架构。

1. *平移不变性*（translation invariance）：不管检测对象出现在图像中的哪个位置，神经网络的前面几层应该对相同的图像区域具有相似的反应，即为“平移不变性”。
2. *局部性*（locality）：神经网络的前面几层应该只探索输入图像中的局部区域，而不过度在意图像中相隔较远区域的关系，这就是“局部性”原则。最终，可以聚合这些局部特征，以在整个图像级别进行预测。

让我们看看这些原则是如何转化为数学表示的。


## 多层感知机的限制
在以往的实验中，对于 FashionMINIST 这个数据集，我们只是简单的将这个变量转换为一个一维张量处理。
现在，我们直接将 `input` 变量保持为二维张量，以保留图像的空间结构信息。

考虑多层感知机的思想：

假设我们的输入为一个大小为 $(h, w)$ 的图像，其中 $h$ 和 $w$ 分别表示图像的高度和宽度。\
在第一层隐藏层过后，我们期望的输出依然是一个二维张量，大小为 $(h', w')$，其中 $h'$ 和 $w'$ 是经过卷积操作后的新高度和宽度。（类比一维的 `input` 与 `output` 长度从 $l$ 扩展（缩小）为 $l'$）

那么，我们的隐藏层的权重参数应该为一个四维张量 $W$，形状为 $(h',w',h,w)$。

输出的二维张量，对于 $(i,j)$ 位置的元素，其结果为：
$$\begin{aligned} \left[\mathbf{H}\right]_{i, j} &= [\mathbf{U}]_{i, j} + \sum_k \sum_l[\mathsf{W}]_{i, j, k, l}  [\mathbf{X}]_{k, l}\\ &=  [\mathbf{U}]_{i, j} +
\sum_a \sum_b [\mathsf{V}]_{i, j, a, b}  [\mathbf{X}]_{i+a, j+b}.\end{aligned}$$

其中，从$\mathsf{W}$到$\mathsf{V}$的转换只是形式上的转换，可以看作张量 $V$ 是张量 $W$ 的一个重新排列。因为在这两个四阶张量的元素之间存在一一对应的关系。
我们只需重新索引下标$(k, l)$，使$k = i+a$、$l = j+b$，由此可得$[\mathsf{V}]_{i, j, a, b} = [\mathsf{W}]_{i, j, i+a, j+b}$。
索引$a$和$b$通过在正偏移和负偏移之间移动覆盖了整个图像。


### 平移不变性
根据平移不变性的要求，我们可以对上述公式做如下改造：

对象在输入$\mathbf{X}$中的平移，应该仅导致隐藏表示$\mathbf{H}$中的平移。也就是说，$\mathsf{V}$和$\mathbf{U}$实际上不依赖于$(i, j)$的值，即$[\mathsf{V}]_{i, j, a, b} = [\mathbf{V}]_{a, b}$。并且$\mathbf{U}$是一个常数，比如$u$。因此，我们可以简化$\mathbf{H}$定义为：
$$[\mathbf{H}]_{i, j} = u + \sum_a\sum_b [\mathbf{V}]_{a, b} [\mathbf{X}]_{i+a, j+b}.$$

这就是*卷积*（convolution）。我们是在使用系数$[\mathbf{V}]_{a, b}$对位置$(i, j)$附近的像素$(i+a, j+b)$进行加权得到$[\mathbf{H}]_{i, j}$。
注意，$[\mathbf{V}]_{a, b}$的系数比$[\mathsf{V}]_{i, j, a, b}$少很多，因为前者不再依赖于图像中的位置。这就是显著的进步！

> 这里做一个简单解释，对于$[\mathsf{V}]_{i, j, a, b} = [\mathbf{V}]_{a, b}$的操作，我们可以简单理解成对于四维张量 $\mathbf{V}$ 的每一个位置上
> $(i,j)$ 的二维矩阵 $v_{i,j}$ 全都相同，也就是他们都共享一个权重矩阵参数。

### 局部性

根据局部性，我们可以对上述公式进一步改造：

如上所述，为了收集用来训练参数$[\mathbf{H}]_{i, j}$的相关信息，我们不应偏离到距$(i, j)$很远的地方。这意味着在$|a|> \Delta$或$|b| > \Delta$的范围之外，我们可以设置$[\mathbf{V}]_{a, b} = 0$。因此，我们可以将$[\mathbf{H}]_{i, j}$重写为

$$[\mathbf{H}]_{i, j} = u + \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.$$

:eqlabel:`eq_conv-layer`

简而言之， :eqref:`eq_conv-layer`是一个*卷积层*（convolutional layer），而卷积神经网络是包含卷积层的一类特殊的神经网络。
在深度学习研究社区中，$\mathbf{V}$被称为*卷积核*（convolution kernel）或者*滤波器*（filter），亦或简单地称之为该卷积层的*权重*，通常该权重是可学习的参数。
当图像处理的局部区域很小时，卷积神经网络与多层感知机的训练差异可能是巨大的：以前，多层感知机可能需要数十亿个参数来表示网络中的一层，而现在卷积神经网络通常只需要几百个参数，而且不需要改变输入或隐藏表示的维数。
参数大幅减少的代价是，我们的特征现在是平移不变的，并且当确定每个隐藏活性值时，每一层只包含局部的信息。


## 卷积

在进一步讨论卷积之前，我们首先要理解一下为什么上面的操作叫做卷积。
实际上，卷积是数学中的概念，它被定义为：
 
$$
(f * g)(t) = \int_{-\infty}^{\infty} f(\tau) g(t - \tau) d\tau
$$

对于二维张量，则为$f$的索引$(a, b)$和$g$的索引$(i-a, j-b)$上的对应加和：

$$(f * g)(i, j) = \sum_a\sum_b f(a, b) g(i-a, j-b).$$
:eqlabel:`eq_2d-conv-discrete`

对于卷积的进一步理解，这里不再给出解释，可以观看[这个视频](https://www.bilibili.com/video/BV1VV411478E)中UP主对CNN中卷积的解释
